# Topic 1A: NumPy for Financial Data
**Module 1 - Introduction to Machine Learning in Python**

This notebook demonstrates core NumPy operations used in financial data analysis,
including array operations, portfolio mathematics, risk metrics, and Monte Carlo simulation.


## 1. Array Creation and Basic Operations


In [1]:
import numpy as np

# Creating arrays from lists
prices = np.array([150.25, 152.30, 148.90, 155.10, 153.80])
print(f'Prices: {prices}')
print(f'Shape: {prices.shape}, Dtype: {prices.dtype}')

# Creating structured arrays
zeros = np.zeros((3, 4))          # 3x4 matrix of zeros
ones = np.ones((2, 5))            # 2x5 matrix of ones
sequence = np.arange(0, 1, 0.1)   # 0.0, 0.1, 0.2, ..., 0.9
linspace = np.linspace(0, 100, 11) # 11 evenly spaced points from 0 to 100

print(f'Zeros shape: {zeros.shape}')
print(f'Linspace: {linspace}')


Prices: [150.25 152.3  148.9  155.1  153.8 ]
Shape: (5,), Dtype: float64
Zeros shape: (3, 4)
Linspace: [  0.  10.  20.  30.  40.  50.  60.  70.  80.  90. 100.]


## 2. Vectorized Operations vs. Loops
Always prefer NumPy vectorized operations over Python loops for performance.


In [6]:
import time

# Generate a large array (1 million daily returns)
np.random.seed(42)
returns = np.random.normal(0.0005, 0.02, 1_000_000)

# SLOW: Python loop to compute cumulative product
start = time.time()
cum_return_loop = 1.0
for r in returns:
    cum_return_loop *= (1 + r)
loop_time = time.time() - start

# FAST: NumPy vectorized
start = time.time()
cum_return_numpy = np.prod(1 + returns)
numpy_time = time.time() - start

print(f'Loop result:  {cum_return_loop:.6f}  Time: {loop_time:.4f}s')
print(f'NumPy result: {cum_return_numpy:.6f}  Time: {numpy_time:.4f}s')
print(f'NumPy is {loop_time/numpy_time:.0f}x faster')


Loop result:  219028124722073963796963838016083144519339789279964415912579968771842212203146117405009940409559777379435962924269568.000000  Time: 0.2487s
NumPy result: 219028124722073963796963838016083144519339789279964415912579968771842212203146117405009940409559777379435962924269568.000000  Time: 0.0031s
NumPy is 81x faster


## 3. Indexing, Slicing, and Boolean Masking


In [7]:
# Daily returns for 5 stocks over 20 days
np.random.seed(42)
returns = np.random.normal(0.001, 0.02, (20, 5))
stock_names = ['JPM', 'GS', 'BAC', 'MS', 'C']

# Slicing: first 10 days, first 3 stocks
subset = returns[:10, :3]
print(f'Subset shape: {subset.shape}')

# Boolean masking: find days where JPM had positive returns
jpm_positive = returns[:, 0] > 0
print(f'JPM positive days: {jpm_positive.sum()} out of {len(jpm_positive)}')

# Filter: all returns on days where JPM was positive
positive_day_returns = returns[jpm_positive]
print(f'Returns on JPM-positive days: {positive_day_returns.shape}')

# Fancy indexing: select specific stocks (JPM and BAC)
selected = returns[:, [0, 2]]
print(f'Selected stocks shape: {selected.shape}')


Subset shape: (10, 3)
JPM positive days: 10 out of 20
Returns on JPM-positive days: (10, 5)
Selected stocks shape: (20, 2)


## 4. Portfolio Returns and Risk Metrics


In [ ]:
# Daily returns for 3 assets over 252 trading days (1 year)
np.random.seed(42)
returns = np.random.normal(0.001, 0.02, (252, 3))  # mean=0.1% daily, std=2%

# Portfolio weights (must sum to 1)
weights = np.array([0.50, 0.30, 0.20])
assert np.isclose(weights.sum(), 1.0), 'Weights must sum to 1'

# Portfolio daily returns using matrix multiplication
portfolio_returns = returns @ weights  # Shape: (252,)
print(f'Portfolio returns shape: {portfolio_returns.shape}')

# Annualized return
annualized_return = portfolio_returns.mean() * 252
print(f'Annualized Return: {annualized_return:.4f} ({annualized_return*100:.2f}%)')

# Annualized volatility
annualized_vol = portfolio_returns.std() * np.sqrt(252)
print(f'Annualized Volatility: {annualized_vol:.4f} ({annualized_vol*100:.2f}%)')

# Sharpe Ratio (assuming risk-free rate = 0)
sharpe = annualized_return / annualized_vol
print(f'Sharpe Ratio: {sharpe:.4f}')

# Value-at-Risk (95% confidence)
var_95 = np.percentile(portfolio_returns, 5)  
print(f'Daily 95% VaR: {var_95:.4f} ({var_95*100:.2f}%)')

# Conditional VaR (Expected Shortfall) - average loss beyond VaR
cvar_95 = portfolio_returns[portfolio_returns <= var_95].mean()
print(f'Daily 95% CVaR: {cvar_95:.4f} ({cvar_95*100:.2f}%)')


Portfolio returns shape: (252,)
Annualized Return: 0.2112 (21.12%)
Annualized Volatility: 0.1820 (18.20%)
Sharpe Ratio: 1.1604
Daily 95% VaR: -0.0161 (-1.61%)
Daily 95% CVaR: -0.0218 (-2.18%)


## 5. Correlation and Covariance Matrices


In [7]:
# Correlation matrix
corr_matrix = np.corrcoef(returns.T)  # Transpose: each row = one asset
print('Correlation Matrix:')
for i, name in enumerate(['Asset_1', 'Asset_2', 'Asset_3']):
    print(f'  {name}: {corr_matrix[i].round(4)}')

# Covariance matrix (annualized)
cov_matrix = np.cov(returns.T) * 252
print(f'\nAnnualized Covariance Matrix shape: {cov_matrix.shape}')

# Portfolio variance using matrix formula: w' * Cov * w
port_variance = weights @ cov_matrix @ weights
port_vol = np.sqrt(port_variance)
print(f'Portfolio Volatility (from cov matrix): {port_vol:.4f}')


Correlation Matrix:
  Asset_1: [ 1.      0.0389 -0.0326]
  Asset_2: [ 0.0389  1.     -0.0782]
  Asset_3: [-0.0326 -0.0782  1.    ]

Annualized Covariance Matrix shape: (3, 3)
Portfolio Volatility (from cov matrix): 0.1824


## 6. Monte Carlo Simulation for Portfolio Optimization


In [8]:
n_simulations = 10_000
n_assets = 3

# Store results
all_weights = np.zeros((n_simulations, n_assets))
all_returns = np.zeros(n_simulations)
all_volatilities = np.zeros(n_simulations)
all_sharpes = np.zeros(n_simulations)

for i in range(n_simulations):
    # Random weights that sum to 1
    w = np.random.random(n_assets)
    w /= w.sum()
    all_weights[i] = w
    
    # Portfolio metrics
    port_ret = (returns @ w).mean() * 252
    port_vol = np.sqrt(w @ cov_matrix @ w)
    
    all_returns[i] = port_ret
    all_volatilities[i] = port_vol
    all_sharpes[i] = port_ret / port_vol

# Find the best portfolio
best_idx = all_sharpes.argmax()
print(f'Best Sharpe: {all_sharpes[best_idx]:.4f}')
print(f'Best Return: {all_returns[best_idx]:.4f}')
print(f'Best Vol:    {all_volatilities[best_idx]:.4f}')
print(f'Best Weights: {all_weights[best_idx].round(4)}')


Best Sharpe: 2.3456
Best Return: 0.5002
Best Vol:    0.2133
Best Weights: [6.050e-01 4.000e-04 3.946e-01]


## 7. Rolling Volatility (No Loops)


In [8]:
import matplotlib.pyplot as plt

window = 30
# Compute rolling volatility using list comprehension (acceptable for moderate sizes)
rolling_vol = np.array([
    portfolio_returns[i:i+window].std() * np.sqrt(252)
    for i in range(len(portfolio_returns) - window + 1)
])

plt.figure(figsize=(12, 4))
plt.plot(rolling_vol, color='steelblue', linewidth=1)
plt.axhline(y=rolling_vol.mean(), color='red', linestyle='--', label=f'Mean: {rolling_vol.mean():.4f}')
plt.title('Rolling 30-Day Annualized Volatility', fontsize=14)
plt.xlabel('Trading Day')
plt.ylabel('Annualized Volatility')
plt.legend()
plt.tight_layout()
plt.show()


NameError: name 'portfolio_returns' is not defined